# 02 Embeddings

Builds the two files the app needs from `data/jobs_entry.csv`:

- `data/jobs_entry_skills.csv`: one row per job, with `skills` and `languages` columns (`"; "`-joined strings).
- `data/job_embeddings.npy`: `multi-qa-mpnet-base-cos-v1` embeddings, L2-normalised, **same row order** as the CSV.

**Model choice:** `multi-qa-mpnet-base-cos-v1` was chosen over the more common general-purpose
`all-MiniLM-L6-v2` because this task -- a short resume matched against long job postings -- is
asymmetric retrieval, which this model is trained for; it is trained for cosine similarity
(matching the cosine-similarity approach used in the report, rather than dot-v1's dot-product
objective), and it has a 512-token limit vs MiniLM's 256. Its own training data was capped at
around 250 word pieces per input, even though it can process up to 512, so the title and skills
are still placed first in the embedding text. Section 7 at the end documents the comparison that
justified this choice.

Run with **Restart > Run All**.

# 1. Setup

In [1]:
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

sys.path.append("..")
from modules.config import MODEL_NAME
from modules.job_text import build_texts, extract_skills_and_languages, hash_texts

DATA = Path("../data")
CACHE = Path(".cache") if Path(".cache").exists() else Path("notebooks/.cache")

c:\Users\liewm\OneDrive - UOW Malaysia KDU\FYP1\Grad2Kerja\grad2kerja\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 2. Load entry-level jobs

In [2]:
jobs = pd.read_csv(DATA / "jobs_entry.csv")
print(len(jobs), "postings")
jobs[["job_title", "subcategory", "category"]].head()

10956 postings


,job_title,subcategory,category
0,Procurement Executive (Contract),"Purchasing, Procurement & Inventory","Manufacturing, Transport & Logistics"
1,Account Executive/ Assistant,Bookkeeping & Small Practice Accounting,Accounting
2,Purchasing Executive,"Purchasing, Procurement & Inventory","Manufacturing, Transport & Logistics"
3,Accounts Assistant (AP),Accounts Payable,Accounting
4,Admin Assistant,Administrative Assistants,Administration & Office Support


In [3]:
assert len(jobs) == 10956, f"expected 10956 entry-level jobs, got {len(jobs)} — check jobs_entry.csv is the latest filtered output"

# 3. Extract skills and languages
The reviewed dictionary is matched against each description. Only the tokenizer is needed by the
`PhraseMatcher`, so the other spaCy components are disabled for speed. Terms are sorted so the output is stable.

Implementation lives in `modules/job_text.py` (`extract_skills_and_languages`), shared with `notebooks/encode_mpnet_lean.py` so the two can't drift apart.


In [4]:
t0 = time.time()
jobs = extract_skills_and_languages(jobs)
print(f"Extraction took {time.time() - t0:.1f}s")

Extraction took 222.8s


In [5]:
n_skills = jobs["skills"].map(lambda s: len(s.split("; ")) if s else 0)
print(f"Postings with no skills found: {(n_skills == 0).mean():.1%}")
print(f"Mean skills per posting: {n_skills.mean():.2f}")
jobs[["job_title", "skills", "languages"]].sample(5, random_state=1)

Postings with no skills found: 6.7%
Mean skills per posting: 3.72


,job_title,skills,languages
2492,Sales Admin Support,administration; communication skills; data ent...,bahasa malaysia; english; mandarin
619,Personal Banker ( April 2024 Intake I Nationwi...,banking; sales,
9920,"Assistant, Sales and Marketing Administration ...",microsoft excel; microsoft office; sales,bahasa malaysia; english
9843,Customer Relation Admin Coordinator,marketing; networking; sales,
8496,Marketing Executive ( Fresh Graduate),interpersonal skills; marketing; microsoft exc...,


# 4. Build the embedding text
Format: `"{job_title}. {subcategory}. Skills: {skills}. {description}"`.
The title and skills come first because both models truncate long inputs, so the most useful
information must not be cut off. The app must apply the same L2 normalisation to the resume embedding; the resume does not need to follow this job-text format.

`embedding_text()` and `build_texts()` also live in `modules/job_text.py`, for the same reason.


In [6]:
texts = build_texts(jobs)
print(texts[0][:400])

Procurement Executive (Contract). Purchasing, Procurement & Inventory. Skills: administration; audit; continuous improvement; procurement; purchasing. Position Purpose Manage aspects of procurement including supplier management, contract management, and vendor performance tracking. Support the administration and management of procurement processes. This includes setting up, maintaining, and admini


In [7]:
model = SentenceTransformer(MODEL_NAME)

tok_lens = pd.Series([len(model.tokenizer.encode(t)) for t in texts])
print(tok_lens.describe())
print(f"{(tok_lens > model.max_seq_length).mean():.1%} of jobs exceed {model.max_seq_length} tokens and are truncated")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 482.13it/s]
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (909 > 512). Running this sequence through the model will result in indexing errors


count    10956.000000
mean       329.245801
std        186.651932
min         25.000000
25%        205.000000
50%        287.000000
75%        403.000000
max       2317.000000
dtype: float64
12.6% of jobs exceed 512 tokens and are truncated


# 5. Embed
Encoding all 10,956 jobs with this model inside the notebook -- alongside spaCy, pandas, and
everything else already loaded -- segfaulted (OOM) on this machine's 7.3GB RAM. `notebooks/
encode_mpnet_lean.py` runs the same `encode()` standalone (nothing else resident, batch_size=8) and
caches the result to `notebooks/.cache/` (gitignored, regenerate by re-running that script). This
cell loads that cache; if it's missing, it falls back to encoding live in-notebook (only safe on a
higher-RAM machine, or shrink `batch_size` further).

In [8]:
cache_npy = CACHE / "mpnet_embeddings.npy"
cache_meta = CACHE / "mpnet_encode_meta.json"

if cache_npy.exists() and cache_meta.exists():
    embeddings = np.load(cache_npy)
    meta = json.loads(cache_meta.read_text())
    assert meta.get("model") == MODEL_NAME, (
        f"cache was built with {meta.get('model')!r}, not {MODEL_NAME!r} -- "
        "regenerate via notebooks/encode_mpnet_lean.py"
    )
    assert meta.get("texts_hash") == hash_texts(texts), (
        "cache was built from different job text (different data or a different "
        "skill_dictionary.py) -- regenerate via notebooks/encode_mpnet_lean.py"
    )
    encode_time = meta["encode_time_s"]
    print(f"Loaded cached embeddings: shape {embeddings.shape}, encoded in {encode_time:.1f}s "
          f"at batch_size={meta['batch_size']} (standalone run, see notebooks/encode_mpnet_lean.py)")
    assert len(embeddings) == len(jobs), "cached embeddings don't match current jobs_entry.csv row count"
else:
    print("No cache found -- encoding live in-notebook (needs several GB of free RAM).")
    t0 = time.time()
    embeddings = model.encode(texts, batch_size=8, normalize_embeddings=True, show_progress_bar=True)
    embeddings = embeddings.astype(np.float32)
    encode_time = time.time() - t0
    print(f"Embedding took {encode_time:.1f}s, shape {embeddings.shape}")

Loaded cached embeddings: shape (10956, 768), encoded in 3954.8s at batch_size=8 (standalone run, see notebooks/encode_mpnet_lean.py)


# 6. Save and check

In [9]:
np.save(DATA / "job_embeddings.npy", embeddings)
jobs.to_csv(DATA / "jobs_entry_skills.csv", index=False)

assert len(jobs) == len(embeddings), "row count mismatch"
assert np.allclose(np.linalg.norm(embeddings, axis=1), 1.0, atol=1e-4), "not L2-normalised"
print("Saved", len(jobs), "rows;", embeddings.shape[1], "dimensions")

Saved 10956 rows; 768 dimensions


In [10]:
accounting_query = "Accounting graduate. Skills: accounting; bookkeeping; microsoft excel; accounts payable; communication skills."
q = model.encode(accounting_query, normalize_embeddings=True)
top = np.argsort(-(embeddings @ q))[:5]
jobs.iloc[top][["job_title", "category", "skills"]].assign(score=(embeddings @ q)[top].round(3))

,job_title,category,skills,score
4124,Accountant (Graduate Trainee),Accounting,accounting; accounting principles; administrat...,0.774
8248,Finance Officer / Executive (GL & FA),Accounting,accounting; finance; microsoft excel; microsof...,0.728
8665,Internship - Accounting/Finance,Accounting,accounting; communication skills; finance; mic...,0.727
5132,Account Cum Admin Executive,Accounting,accounting; administration; microsoft excel; m...,0.725
8449,"Executive, Accounts",Accounting,accounting; banking; finance; microsoft excel;...,0.717


In [11]:
ict_query = "Computer science graduate. Built web apps with Python and JavaScript, REST APIs, SQL databases, and Git. Interested in software developer roles."
q = model.encode(ict_query, normalize_embeddings=True)
top = np.argsort(-(embeddings @ q))[:5]
jobs.iloc[top][["job_title", "category", "skills"]].assign(score=(embeddings @ q)[top].round(3))

,job_title,category,skills,score
10895,Software Engineer,Information & Communication Technology,finance; human resources; javascript; php; python,0.738
6791,Software Developer,Information & Communication Technology,c#; html; java; javascript; python; sql,0.720
1637,Software Developer,Information & Communication Technology,c#; html; java; javascript; python; sql,0.717
2413,Software Engineer (Web Application),Information & Communication Technology,c#; html; restful apis,0.688
3238,Graduate Software Engineer / Developer,Information & Communication Technology,application development; c#; html; java; javas...,0.685


# 7. Model selection: why mpnet over all-MiniLM-L6-v2
`all-MiniLM-L6-v2` is a general-purpose sentence-similarity model with a 256-token limit; it was
the original baseline for this project. It's re-run here (live, in-notebook -- this model is small
enough that this is safe on this machine) as the "alternative considered", to document why
`multi-qa-mpnet-base-cos-v1` (used above) was chosen instead.

The choice is justified on design grounds: this is asymmetric retrieval (short resume vs long job
posting), and truncation was far lower with mpnet (12.6% vs 59.1% of postings truncated). Both
models give sensible top-5 results on the two sample queries below -- neither returns an obviously
irrelevant job. Retrieval quality itself (not just truncation and face-value relevance) will be
measured properly on the manually labelled resume-job pairs in the evaluation experiment, rather
than from these two illustrative queries alone.

In [12]:
ALT_MODEL_NAME = "all-MiniLM-L6-v2"
alt_model = SentenceTransformer(ALT_MODEL_NAME)
print("alt max_seq_length:", alt_model.max_seq_length)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2213.70it/s]


alt max_seq_length: 256


In [13]:
alt_tok_lens = pd.Series([len(alt_model.tokenizer.encode(t)) for t in texts])
truncation_comparison = pd.DataFrame({
    "multi-qa-mpnet-base-cos-v1 (chosen)": [model.max_seq_length, (tok_lens > model.max_seq_length).mean()],
    "all-MiniLM-L6-v2 (alternative)": [alt_model.max_seq_length, (alt_tok_lens > alt_model.max_seq_length).mean()],
}, index=["max_seq_length", "truncated_pct"])
truncation_comparison

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (402 > 256). Running this sequence through the model will result in indexing errors


,multi-qa-mpnet-base-cos-v1 (chosen),all-MiniLM-L6-v2 (alternative)
max_seq_length,512.000000,256.000000
truncated_pct,0.125776,0.591274


In [14]:
t0 = time.time()
alt_embeddings = alt_model.encode(
    texts, batch_size=64, normalize_embeddings=True, show_progress_bar=True
).astype(np.float32)
alt_encode_time = time.time() - t0
print(f"MiniLM embedding took {alt_encode_time:.1f}s, shape {alt_embeddings.shape}")

Batches: 100%|██████████| 172/172 [06:17<00:00,  2.20s/it]

MiniLM embedding took 378.0s, shape (10956, 384)


In [15]:
def compare_query(query_text, label):
    """Run one query against both models' embeddings and show the top-5 side by side."""
    q_mpnet = model.encode(query_text, normalize_embeddings=True)
    q_alt = alt_model.encode(query_text, normalize_embeddings=True)

    top_mpnet = np.argsort(-(embeddings @ q_mpnet))[:5]
    top_alt = np.argsort(-(alt_embeddings @ q_alt))[:5]

    cols = ["job_title", "category", "skills"]
    mpnet_df = jobs.iloc[top_mpnet][cols].assign(score=(embeddings @ q_mpnet)[top_mpnet].round(3)).reset_index(drop=True)
    alt_df = jobs.iloc[top_alt][cols].assign(score=(alt_embeddings @ q_alt)[top_alt].round(3)).reset_index(drop=True)

    print(f"=== {label} ===")
    print("--- multi-qa-mpnet-base-cos-v1 (chosen) ---")
    display(mpnet_df)
    print("--- all-MiniLM-L6-v2 (alternative) ---")
    display(alt_df)

In [16]:
compare_query(accounting_query, "Accounting query")

=== Accounting query ===
--- multi-qa-mpnet-base-cos-v1 (chosen) ---


,job_title,category,skills,score
0,Accountant (Graduate Trainee),Accounting,accounting; accounting principles; administrat...,0.774
1,Finance Officer / Executive (GL & FA),Accounting,accounting; finance; microsoft excel; microsof...,0.728
2,Internship - Accounting/Finance,Accounting,accounting; communication skills; finance; mic...,0.727
3,Account Cum Admin Executive,Accounting,accounting; administration; microsoft excel; m...,0.725
4,"Executive, Accounts",Accounting,accounting; banking; finance; microsoft excel;...,0.717


--- all-MiniLM-L6-v2 (alternative) ---


,job_title,category,skills,score
0,ACCOUNTS EXECUTIVE,Accounting,accounting; interpersonal skills; microsoft ex...,0.826
1,Account Associate,Accounting,accounting; accounts payable; bookkeeping; com...,0.794
2,Accounts Executive,Accounting,accounting; autocount; basic computer skills; ...,0.786
3,Accounts Assistant / Fresh Graduates / Internship,Accounting,accounting software; interpersonal skills,0.777
4,Finance Officer (Plastic Injection Moulding/5 ...,Accounting,accounting; accounts payable; communication sk...,0.776


In [17]:
compare_query(ict_query, "ICT query")

=== ICT query ===
--- multi-qa-mpnet-base-cos-v1 (chosen) ---


,job_title,category,skills,score
0,Software Engineer,Information & Communication Technology,finance; human resources; javascript; php; python,0.738
1,Software Developer,Information & Communication Technology,c#; html; java; javascript; python; sql,0.720
2,Software Developer,Information & Communication Technology,c#; html; java; javascript; python; sql,0.717
3,Software Engineer (Web Application),Information & Communication Technology,c#; html; restful apis,0.688
4,Graduate Software Engineer / Developer,Information & Communication Technology,application development; c#; html; java; javas...,0.685


--- all-MiniLM-L6-v2 (alternative) ---


,job_title,category,skills,score
0,"Intern - Digitalisation, Development & Deliver...",Information & Communication Technology,communication skills; html; javascript; python,0.759
1,Graduate Software Engineer / Developer,Information & Communication Technology,application development; c#; html; java; javas...,0.728
2,Software Developer,Information & Communication Technology,c#; html; java; javascript; python; sql,0.726
3,Software Engineer - Full stack Intern,Information & Communication Technology,html; javascript; python; restful apis,0.724
4,Software Developer,Information & Communication Technology,c#; html; java; javascript; python; sql,0.723


## 7 summary

|                              | multi-qa-mpnet-base-cos-v1 (chosen) | all-MiniLM-L6-v2 (alternative) |
|------------------------------|--------------------------------------|----------------------------------|
| max_seq_length               | 512                                  | 256                              |
| Truncated postings           | 12.6%                | 59.1%            |
| Encoding time (10,956 jobs)  | 3954.8s (batch_size=8, standalone script -- see note in section 5) | 378.0s (batch_size=64, live in-notebook) |

